# QROUTE_COMPLETE_EXPERIMENT — quantization-regret-aware precision x resolution routing (one notebook)

**Purpose.** Q-ROUTE is the second pre-registered follow-up. Its design is frozen in `configs/prereg_qroute.yaml`;
the notebook checks that file's SHA-256 and stops if it changed. Q-ROUTE routes each frame to one of six TensorRT
engines (FP16/INT8 x 640/480/320). It picks the engine that minimises *predicted quantization regret + lambda x cost*.
The cost charges the measured engine latency under current GPU contention **and** the controller's own overhead.
r04 (v1) stays the primary pre-registered result. r05 (v2) and Q-ROUTE (r06) are reported whatever they show.

**Data.**
- Development = the 60 clips used so far: training, model selection, and lambda/K tuning.
- Final test = 40 NEW clips that no earlier experiment touched, drawn once from BDD100K MOT val and
  evaluated once with frozen (hashed) configurations.

**Stages (gated; a failed gate writes a `_PARTIAL` zip and stops).** One section per step:
1. Environment
2. Global config
3. Dataset loading (+ optional VisDrone validation)
4. Model preparation (+ INT4 support check)
5. Fixed baselines
6. Counterfactual cache
7. Oracle headroom
8. Feature engineering
9. Regret prediction
10. Q-ROUTE controller
11. Hardware awareness
12. Controller optimization
13. Baseline comparisons
14. Ablations
15. Final held-out evaluation
16. Live benchmarking
17. Figures
18. Export
19. Validation summary

Every number is written to `results/` with the config, seed, git commit, hardware and prereg hash.
Replay values (exact mAP on cached detections, estimated cost) and live measurements are labelled
separately.

**Before running**
1. *Add Data* → **`tcjpaperdatasets`** (BDD100K MOT). Optional: a VisDrone-VID dataset; it is only validated, not
   part of the pre-registered pipeline.
2. *Settings* → **Accelerator: GPU T4 x2** (pinned to one GPU), **Internet: On**; *Add-ons → Secrets* → tick **`GH_TOKEN`**.
3. `SMOKE_TEST = True` first: about 40 min, mostly building 7 engines. Then `SMOKE_TEST = False` →
   *Save Version → Save & Run All*: about 3–4 h. These are estimates, not yet measured. Caches are checkpointed
   per clip inside the session.

**Download afterwards:** `results_QROUTE_COMPLETE_EXPERIMENT.zip` → unzip into the repo root. It contains
`results/bdd100k_mot/r06_*`, `results/microbench/r06_*`, `results/locked/locked_qroute_*` (+ `.sha256`,
predictor pickles), `results/r06_stages/stages.json`, `results/r06_figures/`, `results/r06_tables/` and `logs/`.

## 1. Environment
The next two cells check the GPU and pin it, clone the repo at a clean commit, check the notebook revision and
run the locked installs.


In [ ]:
# ===================== CELL 1: CONFIG + ENVIRONMENT CHECK =====================
# Edit nothing below for a normal run except SMOKE_TEST.
SMOKE_TEST = True   # True: tiny subset (run this first). False: full run.
NOTEBOOK_REV = "59c656d67ac2"   # hash of this notebook's generated source; checked against the cloned repo

CONFIG = {
    "notebook": "QROUTE_COMPLETE_EXPERIMENT",
    "repo_https": "https://github.com/vinaykamble-nsio/adapres.git",       # private repo, cloned with Secret GH_TOKEN
    "repo_ref": "main",
    "repo_dir": "/tmp/adapres",                     # /tmp: not saved into notebook output
    "repo_zip_marker": "adapres/GIT_COMMIT",        # fallback: repo zip attached as a dataset
    "config_path": "configs/default.yaml",          # relative to repo_dir
    "overrides": {                                  # applied on top of default.yaml
        "paths": {"input_root": "/kaggle/input", "work_root": "/kaggle/working"},
    },
    "cuda_visible_devices": "0",                    # T4 x2 -> pin to one GPU
    "required_gpu_substring": "T4",
}

import os, sys, subprocess, json, platform, base64, glob, shutil, time
os.environ["CUDA_VISIBLE_DEVICES"] = CONFIG["cuda_visible_devices"]  # before any CUDA import
T0 = time.time()

def sh(cmd, check=True):
    """Run a shell command, print its output, raise with the real error on failure."""
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.stderr.strip():
        print(r.stderr[-6000:], file=sys.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

sh("nvidia-smi")
print("python", sys.version)
for mod in ("torch", "onnxruntime", "tensorrt", "ultralytics"):
    try:
        m = __import__(mod)
        print(f"{mod} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{mod} NOT AVAILABLE YET ({e!r}); installed in cell 2 if pinned")
import torch
print("torch.cuda.is_available:", torch.cuda.is_available(), "| visible devices:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
gpu = torch.cuda.get_device_name(0)
print("GPU 0:", gpu)
if CONFIG["required_gpu_substring"] not in gpu:
    raise RuntimeError(f"GPU is {gpu!r}; this project requires a T4 (P100 has no fast INT8). "
                       "Settings -> Accelerator -> GPU T4 x2, then restart.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"expected exactly 1 visible GPU after pinning, got {torch.cuda.device_count()}")

# ---- get the repo: private GitHub clone (Secret GH_TOKEN), else attached zip dataset
REPO = CONFIG["repo_dir"]
repo_source = None
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
try:
    if "github.com/OWNER/" in CONFIG["repo_https"]:
        raise RuntimeError("repo_https is still the placeholder: set it in configs/kaggle.yaml and regenerate")
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    if not token:
        raise RuntimeError("Secret GH_TOKEN is empty")
    hdr = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    # token passed as a one-off header: never written to .git/config or printed
    r = subprocess.run(["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {hdr}",
                        "clone", "--quiet", CONFIG["repo_https"], REPO], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed: " + r.stderr.replace(token, "***").replace(hdr, "***"))
    sh(["git", "-C", REPO, "checkout", "--quiet", CONFIG["repo_ref"]])
    repo_source = "github"
    del token, hdr
except Exception as e:
    print(f"GitHub clone unavailable: {e!r}\nTrying zip-dataset fallback ...")
    hits = sorted(glob.glob("/kaggle/input/**/" + CONFIG["repo_zip_marker"], recursive=True))
    if not hits:
        raise RuntimeError("Neither GitHub clone (Secret GH_TOKEN) nor an attached repo zip "
                           "(dataset containing adapres/GIT_COMMIT) is available. See README cell.") from e
    shutil.copytree(os.path.dirname(hits[0]), REPO)
    repo_source = "zip_dataset:" + os.path.dirname(hits[0])

sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "scripts"))
# Re-running cells in a live kernel re-clones the repo but Python would keep the OLD imported
# modules while GIT records the NEW commit. Drop them so the code always matches the hash.
_stale = [m for m in list(sys.modules) if m == "adapres" or m.startswith("adapres.") or m == "make_notebooks"]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"dropped {len(_stale)} previously imported adapres modules (fresh code from this clone)")
import make_notebooks
expected_rev = make_notebooks.notebook_rev(CONFIG["notebook"] + ".ipynb")
if expected_rev != NOTEBOOK_REV:
    raise RuntimeError(f"STALE NOTEBOOK: this notebook is rev {NOTEBOOK_REV} but repo {CONFIG['repo_ref']} "
                       f"generates rev {expected_rev}. Re-import notebooks/{CONFIG['notebook']}.ipynb "
                       "from the repo (File -> Import Notebook), or push your local commits.")
print("notebook rev", NOTEBOOK_REV, "matches repo")
from adapres import config as C
from adapres.repro import git_commit, set_seed
cfg = C.load(os.path.join(REPO, CONFIG["config_path"]), CONFIG["overrides"])
SEED = cfg["seed"]
set_seed(SEED)
GIT = git_commit(REPO)
print("repo source:", repo_source)
print("git commit:", GIT)
print("seed:", SEED, "| SMOKE_TEST:", SMOKE_TEST)
if GIT["commit"] is None:
    raise RuntimeError("could not determine the git commit; refusing to produce unattributable results")
if GIT["dirty"]:
    raise RuntimeError("repo checkout is dirty; results would not match the recorded commit")
RESULTS = C.resolve_out(cfg, "results_dir", SMOKE_TEST)
LOGS = C.resolve_out(cfg, "logs_dir", SMOKE_TEST)
# Start every run with empty results/ and logs/: the final zip packs these folders, and leftovers
# from an earlier run in the same session would otherwise be shipped as if produced now.
# (Resumable caches live in work dirs, not here.)
for d in (RESULTS, LOGS):
    if os.path.isdir(d) and os.listdir(d):
        print(f"clearing stale outputs from an earlier run in this session: {d} ({len(os.listdir(d))} entries)")
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
print("results ->", RESULTS, "| logs ->", LOGS)


In [ ]:
# ===================== CELL 2: LOCKED INSTALLS + VERSION CHECK =====================
# Internet must be ON. Installs only what the Kaggle image lacks; torch is not touched.
# Every pin is exact (locked 2026-11-20); the cell fails if anything installs at another version.
import re, importlib.metadata as md
print("python", sys.version, "| torch", torch.__version__, "| torch CUDA", torch.version.cuda)
REQ = os.path.join(REPO, "requirements-kaggle.txt")
lines = [ln.split("#")[0].strip() for ln in open(REQ)]
opts = [tok for ln in lines if ln.startswith("-") for tok in ln.split()]  # e.g. --extra-index-url URL
print("pip options:", opts)
reqs = [ln for ln in lines if ln and not ln.startswith("-")]
PIP = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"]
# Preflight: resolve each requirement on its own (no install) so ALL failures show in one run.
bad = {}
for req in reqs:
    p = subprocess.run(PIP + ["--dry-run", "--no-deps", "--quiet", *opts, req], capture_output=True, text=True)
    if p.returncode != 0:
        bad[req] = "\n".join(l for l in p.stderr.splitlines() if "ERROR" in l) or p.stderr[-2000:]
    print(f"preflight {req:45s} {'OK' if p.returncode == 0 else 'FAILED'}")
if bad:
    for req, err in bad.items():
        print(f"\n--- {req} ---\n{err}")
    raise RuntimeError(f"{len(bad)} requirement(s) have no wheel for this Python/CUDA: {list(bad)}. "
                       "Paste this whole cell output; the available versions are listed above.")
r = sh(PIP + ["--quiet", "-r", REQ], check=False)
with open(os.path.join(LOGS, f"{CONFIG['notebook']}_pip.log"), "w") as f:
    f.write(r.stdout + "\n--- stderr ---\n" + r.stderr)
if r.returncode != 0:
    raise RuntimeError("pip install failed (real error above).")
importlib.invalidate_caches()
# Version lock: record exactly what got installed so requirements-kaggle.txt can be pinned to it.
names = [re.split(r"[<>=!~\[ ]", q)[0] for q in reqs]
resolved = {n: md.version(n) for n in names}
os.makedirs(os.path.join(RESULTS, "env"), exist_ok=True)
lock = "\n".join(f"{n}=={v}" for n, v in resolved.items())
with open(os.path.join(RESULTS, "env", "pip_resolved.txt"), "w") as f:
    f.write(f"# python {sys.version.split()[0]} torch {torch.__version__} cuda {torch.version.cuda}\n{lock}\n")
print("RESOLVED VERSIONS (copy into requirements-kaggle.txt as exact pins):\n" + lock)
trt_dists = ("tensorrt-cu13", "tensorrt-cu13-bindings", "tensorrt-cu13-libs")
if len({resolved.get(d) for d in trt_dists}) != 1:
    raise RuntimeError(f"TensorRT wrapper/bindings/libs version mismatch: {resolved}")
pinned = {re.split(r"[<>=!~\[ ]", q)[0]: q.split("==")[-1] for q in reqs if "==" in q}
drift = {n: (pinned[n], resolved[n]) for n in pinned if pinned[n] != resolved[n]}
if drift:
    raise RuntimeError(f"installed versions differ from the lock (pinned, installed): {drift}")
try:
    import tensorrt
except ImportError:
    # Diagnose instead of guessing: which top-level modules did each TensorRT dist install?
    for d in trt_dists:
        try:
            tops = sorted({str(f).split("/")[0] for f in (md.files(d) or [])})
            print(f"{d}: top-level entries {tops[:25]}")
        except md.PackageNotFoundError:
            print(f"{d}: NOT INSTALLED")
    raise
import onnx, onnxruntime
print("import OK: tensorrt", tensorrt.__version__, "| onnx", onnx.__version__, "| onnxruntime", onnxruntime.__version__)
print("torch still", torch.__version__, "cuda", torch.version.cuda)


## 2. Global config
The frozen design hash, action space, budgets and seeds, plus the run object that every stage below uses.

In [ ]:
# ===================== SECTION 2: GLOBAL CONFIG + RUN OBJECT =====================
import logging, zipfile
os.environ["YOLO_AUTOINSTALL"] = "False"
logging.basicConfig(level=logging.INFO, force=True, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(os.path.join(LOGS, "06_qroute.log"))])
from adapres.pipeline.full import GateError
from adapres.pipeline.qroute_run import QRun
WORK = os.path.join(cfg["paths"]["work_root"], "smoke" if SMOKE_TEST else "", "work06")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
from ultralytics import YOLO
import ultralytics, cv2
print("ultralytics", ultralytics.__version__, "| opencv", cv2.__version__)
_m = YOLO(cfg["model"]["weights"])
WEIGHTS = os.path.abspath(cfg["model"]["weights"])
NAMES = dict(_m.names)
del _m
run = QRun(cfg, SMOKE_TEST, REPO)
import pandas as pd
pd.set_option("display.width", 250)

def zip_now(tag=""):
    zpath = f"/kaggle/working/results_QROUTE_COMPLETE_EXPERIMENT{'_smoke' if SMOKE_TEST else ''}{tag}.zip"
    base = os.path.dirname(RESULTS)
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for top in (RESULTS, LOGS):
            for root, _, files in os.walk(top):
                for fn in files:
                    full = os.path.join(root, fn)
                    z.write(full, os.path.relpath(full, base))
    print("wrote", zpath)

def stage(fn, *a):
    t = time.time()
    try:
        out = fn(*a)
    except Exception as e:
        print(f"\n*** {fn.__name__} FAILED: {e!r}\n*** writing partial results zip, then stopping ***")
        zip_now("_PARTIAL")
        raise
    print(f"{fn.__name__}: {(time.time()-t)/60:.1f} min | gates so far:",
          {k: ("PASS" if v["pass"] else "FAIL") for k, v in run.stages.items()})
    return out

print("prereg_qroute sha256:", run.prereg_sha)
print("action space:", run.configs, "| reference:", run.ref, "| extra fixed reference:", run.extra)
print("qroute config:", json.dumps({k: v for k, v in cfg["qroute"].items() if k != "smoke"}, indent=1))
if SMOKE_TEST:
    print("SMOKE overrides:", cfg["qroute"]["smoke"])


## 3. Dataset loading
- Development clips come from the committed Step 1 manifests.
- 40 NEW final-test clips are drawn once from the untouched BDD100K MOT val clips, stratified day/night with the Step 1 rule.
- Disjointness gates check that the two sets never overlap.

VisDrone is validated only if attached.

In [ ]:
S3 = stage(run.s3_data)
print("dev clips:", len(S3["dev_clips"]), "| final test clips:", len(S3["final_test_clips"]), S3["final_test_strata"])
print("final test set sha256:", S3["final_test_sha256"])
print("VisDrone:", S3["visdrone"])


## 4. Model preparation
- TensorRT engines are rebuilt in this session for every config; INT8 is calibrated on the committed 500 calib frames.
- Output-consistency check.
- Isolated latency profile.
- INT4 support check.

In [ ]:
S4 = stage(run.s4_models, WEIGHTS, NAMES)
print(pd.DataFrame([{"config": c, "wall ms (median)": run.profile[c]["wall_ms"]["median"],
                     "engine ms (median)": run.profile[c]["infer_ms"]["median"]} for c in run.all_cfgs]).round(3).to_string(index=False))
print("INT4:", S4["int4"])


## 5. Fixed baselines
One decode pass per frame runs **every** config. This builds the counterfactual cache, checkpointed per clip. The table shows each fixed config's accuracy on the development clips.

In [ ]:
S5 = stage(run.s5_caches)
print(S5)
print(pd.DataFrame(run.fixed_dev_table()).round(4).to_string(index=False))


## 6. Counterfactual cache
Per-frame regret = F1(FP16-640) - F1(config), from the cache (development only).

In [ ]:
REG = run.regret_summary()
print(pd.DataFrame(REG).T.round(4).to_string())


## 7. Oracle headroom
On development clips, a budgeted oracle per action space is compared with the best fixed config at the same budget. The oracle uses ground truth, so it is an upper bound.

In [ ]:
S7 = stage(run.s7_headroom)
print("budgets (ms):", {k: round(v, 3) for k, v in S7["budgets_ms"].items()})
for b, o in S7["oracle"].items():
    print(b, "best fixed:", o["best_fixed"], round(100 * o["best_fixed_map"], 2),
          {sp: round(100 * o[sp]["map"], 2) for sp in ("precision_only", "resolution_only", "joint")})


## 8. Feature engineering
- Causal, near-free features come from the detections used at t-1 and t-2.
- Image cues are gated in only if they raise held-out Spearman by the pre-registered margin.
- The cost of each component is measured.

In [ ]:
S8 = stage(run.s8_s9_predictor)
print("image cues gated in:", S8["use_image_cues"], "| CV Spearman light / full:", round(S8["spearman_light"], 4),
      round(S8["spearman_full"], 4), "| gain", round(S8["gate_gain"], 4))
print("feature cost (ms):", {k: round(v, 4) for k, v in S8["feature_cost"].items()})
print("feature usefulness (|ridge coef|):", {k: round(v, 4) for k, v in sorted(S8["feature_usefulness"].items(), key=lambda kv: -kv[1])})


## 9. Regret prediction
- Ridge vs gradient-boosted trees, compared by 5-fold clip-grouped CV on development.
- The trees are exported to numpy and validated against scikit-learn.
- Out-of-fold predicted vs observed regret.

In [ ]:
print("chosen predictor:", S8["chosen"])
print(pd.DataFrame(S8["cv_full" if S8["use_image_cues"] else "cv_light"]).to_string(index=False))
print("held-out Spearman per config:", {k: round(v, 3) for k, v in S8["per_config_oof_spearman"].items()})


## 10. Q-ROUTE controller
The deployable controller minimises regret + lambda x cost / cost_ref, with a deadline, persistence, fallback and a slowdown EMA. It must make identical decisions to the fast closed-loop replay used for tuning.

In [ ]:
S10 = stage(run.s10_controller_check)
print(S10)


## 11. Hardware awareness
- A separate GPU process runs at duty cycles 0 / 0.5 / 0.9; the measured slowdown per config comes from these runs.
- Deadline.
- Isolated GPU-board energy per config (the cost used by the energy-objective ablation).

In [ ]:
S11 = stage(run.s11_hardware)
print(pd.DataFrame(S11["slowdown_ratios"]).round(3).to_string())
print("deadline ms:", round(S11["deadline_ms"], 3), "| energy mJ/frame:", run.energy_cost)


## 12. Controller optimization
lambda is set by bisection to each budget, and K is chosen, on cross-fitted closed-loop development replay. This is done for the main controller, the variants and every ablation. Each lock is hashed together with its predictor.

In [ ]:
S12 = stage(run.s12_controller)
print(pd.DataFrame(S12).T.to_string())


## 13. Baseline comparisons
**The final test clips are evaluated here, once**, with the frozen locks. Policies compared:
- fixed configs;
- best fixed at each budget;
- v1 (r04) and v2 (r05);
- precision-only, resolution-only and hardware-only;
- matched random (10 seeds);
- GT oracle (upper bound).

In [ ]:
S15 = stage(run.s15_final_eval)
F = run.final["metrics"]
def row(n):
    a = F[n]["all"]
    return {"policy": n, "mAP50-95": 100 * a["map50_95"], "mAP50": 100 * a["map50"], "est cost ms": a["mean_cost_ms_est"],
            "switch/100": a["switch_rate_per_100"]}
base = [n for n in F if not n.startswith(("random_B_int8_s", "qroute_abl_", "contention_"))]
print(pd.DataFrame([row(n) for n in sorted(base)]).round(3).to_string(index=False))
print("matched random @B_int8:", {k: round(v, 4) for k, v in run.final["aggregates"]["random_B_int8"].items()})
print("errors:", {k: v for k, v in run.final["info"].items() if isinstance(v, dict) and "error" in v})


## 14. Ablations
Each ablation was re-tuned on development exactly like the full controller (section 12).

In [ ]:
abl = [n for n in F if n.startswith(("qroute_abl_", "qroute_hardware_only", "contention_"))]
print(pd.DataFrame([row(n) for n in sorted(abl)]).round(3).to_string(index=False))


## 15. Final held-out evaluation
Paired cluster bootstrap over test clips (1000 resamples). Gives the pre-registered Q1–Q3 and the replay part of Q5.

In [ ]:
for k, v in run.final["bootstrap"]["pairs"].items():
    m = v["map50_95"]
    print(f"{k:45s} {100*m['mean']:+.2f} pts  CI [{100*m['ci95'][0]:+.2f}, {100*m['ci95'][1]:+.2f}]")
print(json.dumps(S15["hypotheses"], indent=1, default=str))


## 16. Live benchmarking
On pre-decoded test frames:
- latency median and p95, throughput;
- GPU-board energy (E_total primary), memory;
- allocation and controller overhead;
- live-vs-replay agreement;
- live contention runs (Q4, Q5).

In [ ]:
S16 = stage(run.s16_live)
s = run.live["protocol"]["summary"]
print(pd.DataFrame([{"config": k, "lat med ms": v["latency_total_ms_median"]["median"], "lat p95 ms": v["latency_total_ms_p95"]["median"],
                     "fps": v["throughput_fps"]["median"] if v["throughput_fps"] else None,
                     "E_total mJ/frame": v["E_total_mJ_per_frame"]["median"] if v["E_total_mJ_per_frame"] else None}
                    for k, v in s.items()]).round(3).to_string(index=False))
print("memory:", run.live["memory"], "| live vs replay:", run.live["live_vs_replay"])
print(pd.DataFrame({k: {kk: v[kk] for kk in ("miss_rate", "latency_median_ms", "latency_p95_ms", "map50_95_first_pass")}
                    for k, v in run.live["contention"].items()}).T.round(4).to_string())
print(json.dumps(S16, indent=1, default=str))


## 17. Figures
Rendered from the `results/` records only:
- accuracy-latency and accuracy-energy Pareto;
- oracle headroom;
- predicted vs observed regret;
- per-scene;
- overhead;
- allocation;
- contention;
- ablations;
- bootstrap statistics.

In [ ]:
sys.path.insert(0, os.path.join(REPO, "paper"))
import make_qroute
FIGS = make_qroute.build(RESULTS, os.path.join(RESULTS, "r06_figures"), os.path.join(RESULTS, "r06_tables"), allow_smoke=SMOKE_TEST)
print(FIGS["figures"])


## 18. Export
The experiment summary (stages, hypotheses, measured vs estimated) is written to `results/`. The final cell zips results and logs.

In [ ]:
SUMMARY = run.s19_summary()
n_files = sum(len(f) for _, _, f in os.walk(RESULTS))
print("files under results/:", n_files)


## 19. Validation summary
What passed, what is incomplete, which hypotheses are supported, and what was NOT run.

In [ ]:
for k, v in run.stages.items():
    print(("PASS " if v["pass"] else "FAIL ") + k)
    for c in v["checks"] + v["soft_checks"]:
        if not c["pass"]:
            print("    not met:", c["check"], str(c.get("detail", ""))[:200])
print("incomplete stages:", SUMMARY["incomplete"])
print("hypotheses supported:", SUMMARY["hypotheses"])
print("measured vs estimated:", json.dumps(SUMMARY["measured_vs_estimated"], indent=1))
print("not run in this notebook: INT4 engines (unsupported, see r06_models/int4.json); Jetson/embedded devices; "
      "VisDrone end-to-end (validated only, if attached)")


In [ ]:
# ===================== FINAL CELL: ZIP RESULTS =====================
import zipfile
suffix = "_smoke" if SMOKE_TEST else ""
zpath = f"/kaggle/working/results_QROUTE_COMPLETE_EXPERIMENT{suffix}.zip"
base = os.path.dirname(RESULTS)  # contains results/ and logs/
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in (RESULTS, LOGS):
        for root, _, files in os.walk(top):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, base))
                n += 1
print(f"wrote {zpath} ({n} files, {os.path.getsize(zpath)/1e6:.2f} MB)")
print(f"total notebook time: {(time.time()-T0)/60:.1f} min")
